# Querying FHIR with SQL 

FHIR data can be queried using SQL to create tabular views of the information stored on FHIR servers. However, these tables need to be manually curated to fill the tables with the information you required. For more information, see the relevant pages of the [documentation](https://docs.intersystems.com/irisforhealth20251/csp/docbook/DocBook.UI.Page.cls?KEY=HXFHIRFSB_installation).

Start by going to the management portal at http://localhost:32783/csp/sys/UtilHome.csp 

On the left-hand side, Click `Health`, then click `DEMO`, then if its not selected, select `FHIR Server Management` from the dropdown, and click the `Go` button

![image](./images/Opening-fhir-server-manager.png)


From here you can see the URL of the FHIR server (red), the number of Resources (pink), and if you click on resources, you can see the number of each resource type. 

![image](./images/FHIR-server-manager.png)

For now though, click the link on the FHIRSQL builder on the left hand side (orange in the image above).



If you are lost at this point, you can always follow this link: http://localhost:32783/csp/fhirsql/index.html#/home . 

There are three sections to this - first we create an analysis, this connects to our FHIR server, then we create a transformation sepecification - this defines which bits of data get added to the SQL table projection, then we create a projection based on this transformation specification. 

### Creating SQL Analyses


Next to Analyses, click the blue `+ New` button and a form will open. 

Next to the `FHIR Respository` dropdown click `+ New` and fill in the following details:

- Name: SQLBuilderConfig (Or choose your own name)
- Host: localhost
- Port: 52773  // This refers to the docker container port, not your local port, so don't change this

- URL: Leave Blank
- SSL: Leave Blank
- Credentials:
    - Click `+ New`, create new credentials profile
        - Username: _SYSTEM
        - Password: ISCDEMO

![Creating Analyses](./images/CreatingAnalyses.png)

If you've filled this out correctly, when you click the 'FHIR Repository URL' dropdown, an option should appear. Select this option and click save. 

When you return to the `New FHIR Analysis` Page enter "100" into the `selectivity percentage` (this just allows you analyse a subset of your data, as we have a small dataset, we will leave it as 100%)


## Creating Transformation Specifications

Click New to create a new transformation specification. Give it a name (If you are making multiple specifications its useful to name them something informative, but feel free to leave it as "demo"), and choose to use the analysis created in the previous step.

A transformation specification controls how your FHIR resources will be put into a table, i.e. which pieces of information within the FHIR data will be mapped into relational tables. 

### Choosing data
At this point, we need to choose each piece of information we would like to make available using SQL. Lets start by adding the patient's Name and ID:

- Scroll to the Patient resource in the navegator on the left hand side. Unpack Patient and Name, then click family. 

![patient dropdown](images/PatientDropdown.png)

The following page will come up: 

![patient name family](images/PatientNameFamily.png)

- You can click 'Show Histogram' to see some surnames in the data. The bar represents the frequency of the name. With our small dataset, its expected that there are few duplicated surnames, but this histogram can be helpful to see if the data is as you expect. 

![patient family name histogram](images/NameHistogram.png)

- We can change the output column name and add the column to the projection below this. In this example, you may want to change the column name to something like "Surname". It may be important to change the name to make it clearer what the data is, for example renaming an ID to PatientID to differentiate it from an ID of an individual resource. 

![Column name and Add to Projection](images/ColumnNameAndAddToProjection.png)

Its possible to do some basic filtering at this step, as well as changing the max length of the value, setting the value as the table index and storing a copy of the data, so you are not editing the real data on the server. We are not going to do of these in this tutorial, but it may be useful for your solutions. 



Once you have clicked `Add to projection`, you can see it added at the bottom of the page, under Transformation Specification columns:

![TransformationSpecificationColumns](images/TransformationSpecificationColumns.png)

You can see we can find the information stored in the column `Surname` of the table `Patient`. 


### Add the rest of the data

For each of the following, open the resource value and click `Add to projection` (or `update`) at the bottom right of the right hand panel. This will add it to the output projection. 

    // This is the reference to the patient
    - DocumentReference -> Subject -> Reference
    
    // This is the encoded clinical note
    - DocumentReference -> Content -> Attachment -> Data 

    // Encounter date
    - DocumentReference -> Date

Also, we might want patient information later, so lets also add some patient details: 
    
    - Patient -> names -> given
    - Patient -> identifier -> value

When you are finished, click `Done` at the top right of the screen. 


### Create Projection 

Finally, you have to create a projection based on the specification you have made. This is as simple as picking the FHIR server and transformation specification we just created from the dropdown menus, then give it a name, this will be the base name for all of your tables. Give the project the name `VectorSearchApp` (feel free to change it, just remember to change SQL queries later). 

![Creating Projection](./images/CreatingProjection.png)

Click `Launch Projection` 


## Querying the database

Now that we have created projections of our data, we can query it through SQL. The easiest method for this is to use the [Management Portal SQL Editor](http://localhost:32783/csp/sys/exp/%25CSP.UI.Portal.SQL.Home.zen?$NAMESPACE=DEMO). 

You may need to change the namespace to the DEMO namespace, this can be done from the top of the page (click the namespace name, shown in the red square in the image below).

You can find the tables created in the list on the left hand side - if you've also called it VectorSearchApp it will be right at the bottom because its alphabetically ordered. You can also search for it. Drag the table into the `Execute Query` box to select all the columnns.

![querying database](./images/QueryingDatabase.png)


## Querying the database with Python 

Now we know that the SQL works, we can query the database with Python. There are several different methods to execute SQL queries from Python, we are going to use the [Python DB-API method](https://docs.intersystems.com/irisforhealth20251/csp/docbook/Doc.View.cls?KEY=BPYNAT_pyapi), as it is the prefered method for using relational querying from a Python Application. This uses the package `intersystems-irispython`, which can be installed with pip:

    pip install intersystems-irispython

To use this method we take the following steps: 

    1. Create a connection
    2. Create a Cursor object
    3. Create the SQL query (and parameters) 
    4. Execute the query from the cursor object
    5. Iterate through the results from the cursor object. 
    6. Extract the results into a pandas DataFrame

Here, its important to note that the cursor execution command does not directly return the results, instead the results are stored within the cursor object.

### Create Connection and Cursor

In [ ]:
pip install intersystems-irispython

In [2]:
## Import IRIS python DB-API Driver
import iris

In [4]:
## Credentials: 
server_location = "localhost"
port_number = 32782
namespace = "DEMO"
user_name = "_SYSTEM"
password = "ISCDEMO"

## Create a connection
conn = iris.connect(server_location, port_number, namespace, user_name, password)

## Create a cursor object
cursor = conn.cursor()

### Create SQL query

There are two options here, we can either create a single query string containing all the search parameters, or we can leave out some parameters to be defined at the point of execution, leaving a '?' character as a placeholder. 

For example, we could run:

    cursor.execute("SELECT col1, col2 FROM exampleTable") 

or we could run:

    cursor.execute("SELECT ?, ? FROM exampleTable", ["col1", "col2"])

These options have the same result, but the second option allows different parameters to be passed into the same query. You can even execute the query with a list of parameter lists using the `cursor.executemany()` command. This functionality is particularly useful when inserting new data into tables. 

For now though, we are going to execute the query I gave above using the syntax from the first example:

In [8]:
sql = """SELECT 
DocumentReferenceContentAttachmentData, DocumentReferenceSubjectReference
FROM VectorSearchApp.DocumentReference"""

In [9]:
cursor.execute(sql)

-1

### Extracting Data

While the result of the query above (-1) may suggest failure, here it actually means the query has been executed successfully. The results can be collected from the cursor object using one of the following commands: 

- `cursor.fetchone()` returns the next row of data from the query.
- `cursor.fetchmany(n)` returns the next n rows of data (where n is an integer). 
- `cursor.fetchall()` returns all the results. 

You can find more specific and useful methods in the [DB-API documentation](https://docs.intersystems.com/irisforhealth20251/csp/docbook/Doc.View.cls?KEY=BPYNAT_pyapi)

We are going to use `cursor.fetchall()` and collect it in a pandas DataFrame. 


In [10]:
result_set = cursor.fetchall()

In [11]:
# Let's look at the first row of the data:
print(result_set[0])

('4f7469746973204d65646961204576616c756174696f6e0a446174653a20323032342d30312d32350a50726f76696465723a2044722e204c656d75656c3330342053746f6b65733435330a4c6f636174696f6e3a20426574682049737261656c20446561636f6e65737320486f73706974616c20e28093204e65656468616d0a526561736f6e20666f722056697369743a20456172207061696e20616e642069727269746162696c6974790a5375626a6563746976653a0a4175726f72612070726573656e74656420776974682073796d70746f6d73206f662065617220646973636f6d666f72742c206d696c642066657665722c20616e6420696e637265617365642066757373696e6573732e20506172656e74207265706f727473206f6e736574203220646179732061676f2e204e6f20766f6d6974696e67206f722064696172726865612e204e6f207072696f7220686973746f7279206f662065617220696e66656374696f6e732e0a4f626a6563746976653a0a0a566974616c733a2054656d702033372e38c2b0432c204250203130372f3830206d6d48670a506879736963616c204578616d3a0a0a54796d70616e6963206d656d6272616e653a204572797468656d61746f757320616e642062756c67696e67206f6e2074686520726967687420736964650a4e6f2064697363

We will use `pandas` to work with the dataset in python. If you haven't already, install pandas  in your environment: 

In [ ]:
pip install pandas

In [12]:
import pandas as pd
## The result_set doesn't include column names so we will add them ourselvs
cols = ["ClinicalNotes", "Patient"] 

df = pd.DataFrame(result_set, columns=cols)

In [13]:
print(f"We have retrived {len(df)} Document Reference Resources.")
df.head()

We have retrived 51 Document Reference Resources.


,ClinicalNotes,Patient
0,4f7469746973204d65646961204576616c756174696f6e...,Patient/3
1,446174653a20323032352d30382d30360a50726f766964...,Patient/3
2,466f6c6c6f772d557020666f72204f7469746973204d65...,Patient/3
3,446174653a20323032342d31312d32310a50726f766964...,Patient/3
4,446174653a20323032342d30382d30360a50726f766964...,Patient/3


The clinical notes shown here are currently encoded. We will decode them in the next tutorial. 

### Next steps

Now we can access the FHIR resources with SQL calls, we can continue with the main tutorial by [Creating a Vector Database](./2-Creating-Vector-DB.ipynb), or you can create SQL datasets for your own project. 